# PydanticAI 입문 가이드

이 노트북은 `analysis_agent.ipynb`, `automation_agent.ipynb`, `planner_agent.ipynb`에서 사용하는 **PydanticAI**의 핵심 개념을 단계별로 설명합니다.

## PydanticAI란?

PydanticAI는 LLM 기반 애플리케이션을 만들기 위한 Python 프레임워크입니다.

핵심 아이디어는 간단합니다:
1. **Agent** — LLM을 감싸는 객체. "누구에게 어떤 일을 시킬지" 정의
2. **Instructions** — Agent에게 주는 시스템 프롬프트 (역할, 규칙)
3. **Structured Output** — LLM 응답을 Pydantic 모델로 강제하여 JSON 형태로 받기
4. **Tools** — Agent가 호출할 수 있는 Python 함수 (검색, API 호출 등)

아래에서 하나씩 실습해보겠습니다.

## 1단계: 환경 설정

먼저 API 키를 로드하고 필요한 라이브러리를 임포트합니다.

- `dotenv`: `.env` 파일에 저장된 `OPENAI_API_KEY`를 환경변수로 로드
- `pydantic`: 구조화된 출력 스키마를 정의하는 데이터 검증 라이브러리
- `pydantic_ai`: LLM Agent를 만들고 실행하는 프레임워크

In [1]:
# .env 파일에서 OPENAI_API_KEY 환경변수 로드
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel   # 구조화된 출력 스키마 정의용
from pydantic_ai import Agent    # LLM 기반 Agent 프레임워크

## 2단계: 가장 간단한 Agent

PydanticAI에서 Agent를 만드는 최소 코드는 단 2줄입니다.

```python
agent = Agent("openai:gpt-5.4")          # 1) Agent 생성 (사용할 LLM 지정)
result = await agent.run("안녕하세요!")    # 2) 실행
```

- `Agent(모델명)`: 어떤 LLM을 사용할지 지정합니다
- `await agent.run(프롬프트)`: Agent에게 메시지를 보내고 응답을 받습니다
- `result.output`: 응답 텍스트가 담겨 있습니다

> **주의:** Jupyter 노트북에서는 `await agent.run()`을 사용합니다.
> 일반 Python 스크립트에서는 `agent.run_sync()`를 사용하면 됩니다.

---

# 실습1

In [2]:
agent = Agent("openai:gpt-5.4")    # 1) Agent 생성 *사용할 LLM 모델 지정
result = await agent.run("포스코DX는 어떤 회사야?")  # 2) Agent 실행 *프롬프트 입력
print(result)

AgentRunResult(output='포스코DX는 **포스코그룹의 IT·자동화·스마트팩토리 전문 계열사**예요.\n\n간단히 말하면,  \n**기업의 디지털 전환(DX: Digital Transformation)** 을 돕는 회사라고 보면 됩니다.  \n회사 이름의 “DX”도 여기서 온 거예요.\n\n### 무슨 일을 하나?\n주요 사업은 대체로 이런 분야예요:\n\n- **스마트팩토리 구축**\n  - 공장의 생산설비를 자동화하고\n  - 데이터를 수집·분석해서\n  - 생산성, 품질, 안전성을 높이는 시스템 제공\n\n- **산업자동화**\n  - 제철소, 발전소, 물류시설 같은 산업 현장에\n  - 제어시스템, 설비 자동화, 엔지니어링 서비스 제공\n\n- **IT 서비스**\n  - 기업용 시스템 구축\n  - 클라우드, AI, 빅데이터, 보안, ERP 같은 디지털 기술 지원\n\n- **스마트 물류 / 스마트 인프라**\n  - 물류 자동화\n  - 항만, 철도, 교통 등 인프라 분야의 디지털 솔루션 제공\n\n### 포스코그룹 안에서는 어떤 역할?\n포스코DX는 포스코그룹 내에서  \n**제철·이차전지·에너지·건설 등 여러 사업의 디지털 전환과 자동화**를 담당하는 핵심 회사 역할을 해요.\n\n즉,\n- 포스코의 공장을 더 똑똑하게 만들고\n- 설비를 자동화하고\n- 데이터를 활용해 운영 효율을 높이는 일을 많이 한다고 보면 됩니다.\n\n### 원래 이름은?\n예전에는 **포스코ICT**라는 이름이었고,  \n디지털 전환 중심 회사라는 이미지를 강화하기 위해 **포스코DX**로 사명을 변경했어요.\n\n### 한 줄로 정리\n**포스코DX = 포스코그룹의 스마트팩토리·자동화·IT 기반 디지털 전환 전문 회사**\n\n원하시면 제가 이어서  \n**“포스코DX의 사업구조/매출원/취업 관점에서 어떤 회사인지”**도 쉽게 설명해드릴게요.')


---

In [3]:
# 가장 간단한 Agent: 모델만 지정하고 바로 실행
simple_agent = Agent("openai:gpt-5.4")

# Agent에게 메시지를 보내고 응답 받기
result = await simple_agent.run("PydanticAI를 한 줄로 설명해줘.")

# result.output에 LLM의 응답 텍스트가 담겨 있음
print(result.output)

PydanticAI는 **Pydantic의 타입 검증·구조화 강점을 활용해 LLM 애플리케이션을 더 안전하고 예측 가능하게 만드는 Python AI 에이전트 프레임워크**입니다.


## 3단계: Instructions (시스템 프롬프트)

`instructions` 파라미터로 Agent에게 **역할과 규칙**을 부여할 수 있습니다.
이는 ChatGPT의 "system prompt"와 같은 역할입니다.

```python
agent = Agent(
    "openai:gpt-5.4",
    instructions="너는 친절한 요리사야. 모든 답변을 요리 비유로 해줘."
)
```

`instructions`가 있으면 Agent는 매 요청마다 이 규칙을 따릅니다.

In [4]:
# instructions로 Agent에게 역할 부여
chef_agent = Agent(
    "openai:gpt-5.4",
    instructions="너는 친절한 요리사야. 모든 답변을 요리 비유로 설명해줘. 한국어로 답변.",
)

# 같은 질문이라도 instructions에 따라 답변 스타일이 달라짐
result = await chef_agent.run("소프트웨어 테스트가 왜 중요한가요?")
print(result.output)

소프트웨어 테스트는 요리에서 **음식을 손님상에 내기 전에 맛보고 확인하는 과정**과 아주 비슷해요.

쉽게 말하면, 테스트가 중요한 이유는 이런 것들이에요:

1. **맛이 이상한 음식이 나가지 않게 하려고**
   - 프로그램에도 버그가 있을 수 있어요.
   - 테스트는 국이 너무 짜거나 덜 익은 재료가 없는지 미리 보는 것과 같아요.
   - 손님에게 내기 전에 문제를 잡을 수 있죠.

2. **일관된 품질을 유지하려고**
   - 어떤 날은 맛있고 어떤 날은 별로면 식당 신뢰가 떨어져요.
   - 소프트웨어도 항상 비슷하게 잘 동작해야 해요.
   - 테스트는 “이 레시피대로 만들면 늘 같은 맛이 나는지” 확인하는 과정이에요.

3. **큰 사고를 막으려고**
   - 상한 재료를 모르고 썼다면 손님이 크게 불편해질 수 있어요.
   - 소프트웨어 오류도 데이터 손실, 보안 문제, 서비스 장애 같은 큰 사고로 이어질 수 있어요.
   - 테스트는 주방 위생 점검처럼 사고를 예방해요.

4. **수정할수록 더 망가지는 걸 막으려고**
   - 요리에 새 재료를 넣었더니 원래 좋던 맛까지 망가질 수 있죠.
   - 프로그램도 새로운 기능을 넣다가 기존 기능이 고장날 수 있어요.
   - 테스트는 새 메뉴를 추가해도 기존 인기 메뉴 맛이 그대로인지 보는 것과 같아요.

5. **개발 속도를 오히려 높이려고**
   - 얼핏 보면 테스트가 번거로워 보여도, 나중에 손님 불만 받고 다시 만드는 것보다 훨씬 빨라요.
   - 즉, 조리 중간중간 맛보는 것이 완성 후 전부 폐기하는 것보다 효율적이에요.

한마디로 말하면,  
**소프트웨어 테스트는 음식을 안전하고 맛있게 내기 위한 최종 점검이자, 좋은 주방을 유지하는 기본 습관**이에요.

원하시면 제가 이것도  
- **초보자 버전**
- **면접 답변 버전**
- **실무자 버전**  
으로 나눠서 더 설명해드릴게요.


---

## 실습

In [5]:
# instructions로 Agent에게 역할 부여
yeosu_tourist_agent = Agent(
    "openai:gpt-5.4",
    instructions="너는 대한민국 여수시의 여행 전문가야. 한국어로 답변.",
)

# 같은 질문이라도 instructions에 따라 답변 스타일이 달라짐
yeosu_tourist_result = await yeosu_tourist_agent.run("여수시에 가볼 만한 곳이 어디야? " \
"단 주어진 시간은 오후 1시부터 7시정도야. 관광지, 맛집, 카페 등 다양하게 알려줘.")
print(yeosu_tourist_result.output)

오후 **1시~7시(약 6시간)** 정도면 여수의 핵심만 알차게 즐기기 좋아요.  
여수는 **바다 풍경, 해상 케이블카, 낭만포차, 카페, 해산물**이 강점이라서 **이동 동선을 잘 짜는 것**이 중요합니다.

아래처럼 **관광지 + 맛집 + 카페**를 섞어서 추천드릴게요.

---

# 1. 6시간 기준으로 가기 좋은 대표 코스

## 코스 A. 여수 처음 방문이라면: “핵심 관광 + 바다뷰 + 먹거리”
**추천 대상:** 처음 여수 가는 분, 사진 찍기 좋아하는 분

### 1) 오후 1:00 — 오동도
- 여수 대표 관광지예요.
- 동백나무 숲길과 바다 산책로가 잘 되어 있어서 가볍게 걷기 좋습니다.
- 시간이 많지 않다면 전체를 오래 돌기보다 **입구~용굴~바다 전망 포인트 위주**로 보세요.
- **소요시간:** 1시간~1시간 20분

### 2) 오후 2:20 — 여수해상케이블카
- 여수에서 가장 인기 많은 체험 중 하나예요.
- 바다 위를 건너는 느낌이 좋아서 **짧은 시간에 여수 감성 느끼기 좋음**
- 일반 캐빈도 좋지만, 무서운 거 괜찮으면 **크리스탈 캐빈**도 추천
- **소요시간:** 대기 포함 40분~1시간  
- 주말/성수기엔 대기시간이 길 수 있으니 참고하세요.

### 3) 오후 3:30 — 돌산공원 또는 자산공원 전망
- 케이블카와 연계해서 가기 좋아요.
- 여수항, 바다, 다리 풍경이 한눈에 보여서 사진 찍기 좋습니다.
- 날씨 좋으면 전망이 정말 예뻐요.
- **소요시간:** 30분

### 4) 오후 4:10 — 카페 타임
추천 카페 스타일:
- **바다뷰 카페:** 돌산대교나 바다가 보이는 카페
- **감성 카페:** 이순신광장 주변, 구도심 쪽 작은 카페들
- **추천 포인트:** 여수는 대형 오션뷰 카페가 많아서 쉬어가기 좋음
- **소요시간:** 40분~1시간

### 5) 오후 5:20 — 저녁식사
여수에서 저녁으로 좋은 메뉴:
- **게장백반**
- **서대회무침**
- **새조개샤브샤브**(계절 영향 있음)


---

## 4단계: Structured Output (구조화된 출력)

기본 Agent는 자유 텍스트를 반환합니다. 하지만 실무에서는 **정해진 형식의 데이터**가 필요합니다.

`output_type`에 Pydantic 모델을 지정하면, LLM이 자유 텍스트 대신 **해당 스키마에 맞는 데이터**를 반환합니다.

### 왜 필요할까요?

| 자유 텍스트 | 구조화된 출력 |
|---|---|
| "이 영화는 8점입니다" | `{"title": "...", "score": 8, "genre": "SF"}` |
| 후속 처리 어려움 | DB 저장, API 연동 바로 가능 |

### 사용법

```python
class Movie(BaseModel):        # 1) 원하는 출력 형식을 Pydantic 모델로 정의
    title: str
    score: int

agent = Agent(
    "openai:gpt-5.4",
    output_type=Movie,          # 2) Agent에 output_type으로 지정
)
result = await agent.run(...)
movie = result.output           # 3) result.output이 Movie 타입으로 반환됨
print(movie.title)              #    → 속성으로 바로 접근 가능
```

> **실전 활용:** `analysis_agent.ipynb`에서 `CodeReview` 모델로 코드 리뷰 결과를, `automation_agent.ipynb`에서 `DailyReport` 모델로 일일 보고서를 구조화합니다.

In [6]:
# Step 1) 원하는 출력 형식을 Pydantic 모델로 정의
class MovieReview(BaseModel):
    title: str       # 영화 제목
    score: int       # 평점 (1~10)
    pros: list[str]  # 장점 목록
    cons: list[str]  # 단점 목록


# Step 2) output_type으로 지정 → LLM이 반드시 이 형식으로 응답
review_agent = Agent(
    "openai:gpt-5.4",
    output_type=MovieReview,
    instructions="영화 평론가로서 요청받은 영화를 평가. 한국어로 작성.",
)

# Step 3) 실행 → result.output이 MovieReview 타입
result = await review_agent.run("인터스텔라를 평가해주세요.")
review = result.output

# Pydantic 모델이므로 속성으로 바로 접근 가능
print(f"제목: {review.title}")
print(f"평점: {review.score}/10")
print(f"장점: {', '.join(review.pros)}")
print(f"단점: {', '.join(review.cons)}")

제목: 인터스텔라
평점: 9/10
장점: 광활한 우주를 배경으로 한 압도적인 시청각 경험과 한스 짐머의 음악이 감정을 극대화한다., 하드 SF와 가족 멜로를 결합해 과학적 상상력과 인간적 정서를 동시에 설득력 있게 끌고 간다., 크리스토퍼 놀란 특유의 구조적 연출과 시간에 대한 주제가 강렬한 여운을 남긴다.
단점: 중후반부의 설명적 대사와 설정 전달이 다소 과잉으로 느껴질 수 있다., 감정선이 강한 만큼 일부 관객에게는 지나치게 신파적으로 받아들여질 여지가 있다., 조연 캐릭터들의 서사가 상대적으로 얕아 몇몇 관계가 기능적으로 소비된다.


## 5단계: Tools (도구)

Agent에게 **Python 함수를 도구로 등록**하면, LLM이 필요할 때 해당 함수를 호출할 수 있습니다.

### 왜 필요할까요?

LLM은 혼자서는 다음을 할 수 없습니다:
- 실시간 데이터 조회 (날씨, 주가, DB)
- 외부 시스템 조작 (이메일 발송, 파일 저장)
- 계산이나 코드 실행

Tool을 등록하면 LLM이 **"이 함수를 호출해야겠다"고 판단**하고, 프레임워크가 실제 함수를 실행합니다.

### 사용법

```python
@agent.tool_plain          # 데코레이터로 도구 등록
def add(a: int, b: int) -> str:
    """두 숫자를 더한다."""     # docstring이 LLM에게 도구 설명으로 전달됨
    return str(a + b)
```

- `@agent.tool_plain`: Agent 컨텍스트 없이 독립적으로 동작하는 도구
- **함수 이름과 docstring**이 LLM에게 전달되어, LLM이 언제 이 도구를 쓸지 판단
- **타입 힌트** (`a: int, b: int`)가 LLM에게 파라미터 형식을 알려줌

> **실전 활용:** `automation_agent.ipynb`에서 `fetch_github_issues`, `send_slack_message`를, `planner_agent.ipynb`에서 `search_web`, `write_section`을 도구로 등록합니다.

In [7]:
# Tool이 있는 Agent 만들기: 간단한 계산기 예제
calc_agent = Agent(
    "openai:gpt-5.4",
    instructions="계산이 필요하면 반드시 제공된 도구를 사용하라. 한국어로 답변.",
)


# @agent.tool_plain 데코레이터로 도구 등록
# - 함수 이름(add)과 docstring이 LLM에게 전달됨
# - LLM이 "덧셈이 필요하다"고 판단하면 이 함수를 호출
@calc_agent.tool_plain
def add(a: int, b: int) -> int:
    """두 숫자를 더한다."""
    print(f"  [Tool 호출] add({a}, {b})")
    return a + b


@calc_agent.tool_plain
def multiply(a: int, b: int) -> int:
    """두 숫자를 곱한다."""
    print(f"  [Tool 호출] multiply({a}, {b})")
    return a * b


# Agent 실행 — LLM이 알아서 필요한 도구를 선택하여 호출
result = await calc_agent.run("17과 28을 더하고, 그 결과에 3을 곱해주세요.")
print(f"\n최종 답변: {result.output}")

  [Tool 호출] add(17, 28)
  [Tool 호출] multiply(45, 3)

최종 답변: 결과는 **135**입니다.


---

## 실습

In [ ]:
# Tool이 있는 Agent 만들기: 간단한 계산기 예제
calc_agent = Agent(
    "openai:gpt-5.4",
    instructions="계산이 필요하면 반드시 제공된 도구를 사용하라. 한국어로 답변.",
)


# @agent.tool_plain 데코레이터로 도구 등록
# - 함수 이름(add)과 docstring이 LLM에게 전달됨
# - LLM이 "덧셈이 필요하다"고 판단하면 이 함수를 호출
@calc_agent.tool_plain
def add(a: int, b: int) -> int:
    """두 숫자를 더한다."""
    print(f"  [Tool 호출] add({a}, {b})")
    return a + b


@calc_agent.tool_plain
def multiply(a: int, b: int) -> int:
    """두 숫자를 곱한다."""
    print(f"  [Tool 호출] multiply({a}, {b})")
    return a * b

---

## 6단계: 모두 합치기 — Tool + Structured Output

지금까지 배운 것을 조합하면 **실전에서 쓸 수 있는 Agent**가 됩니다.

이 예제에서는 도시 이름을 받아 날씨를 조회(Tool)하고, 구조화된 여행 추천(Structured Output)을 반환하는 Agent를 만듭니다.

```
사용자: "서울 여행 추천해줘"
   ↓
Agent가 get_weather("서울") Tool 호출
   ↓
LLM이 날씨 정보를 참고해 TravelAdvice 형식으로 응답
   ↓
result.output → TravelAdvice(city="서울", weather="맑음", ...)
```

In [8]:
# --- 구조화된 출력 정의 ---
class TravelAdvice(BaseModel):
    city: str              # 도시 이름
    weather: str           # 현재 날씨
    recommended: bool      # 여행 추천 여부
    tips: list[str]        # 여행 팁 목록


# --- Agent 생성: Tool + Structured Output 조합 ---
travel_agent = Agent(
    "openai:gpt-5.4",
    output_type=TravelAdvice,  # 구조화된 형식으로 응답 강제
    instructions=(
        "여행 가이드로서 도시의 날씨를 조회하고 여행 조언을 제공한다. "
        "반드시 get_weather 도구로 날씨를 확인한 후 답변하라. 한국어로 작성."
    ),
)


# --- Tool 등록: 날씨 조회 (Mock) ---
@travel_agent.tool_plain
def get_weather(city: str) -> str:
    """도시의 현재 날씨 정보를 조회한다."""
    # 실제로는 날씨 API를 호출하지만, 여기서는 Mock 데이터 반환
    mock_data = {
        "서울": "맑음, 22°C, 습도 45%",
        "부산": "흐림, 19°C, 오후 비 예보",
        "제주": "맑음, 24°C, 바람 강함",
    }
    weather = mock_data.get(city, f"{city}: 데이터 없음, 약 20°C 예상")
    print(f"  [Tool 호출] get_weather('{city}') → {weather}")
    return weather


# --- 실행 ---
result = await travel_agent.run("제주도 여행을 계획 중이에요. 지금 가도 될까요?")
advice = result.output

print(f"도시: {advice.city}")
print(f"날씨: {advice.weather}")
print(f"추천: {'✅ 추천' if advice.recommended else '❌ 비추천'}")
print("팁:")
for tip in advice.tips:
    print(f"  - {tip}")

  [Tool 호출] get_weather('제주도') → 제주도: 데이터 없음, 약 20°C 예상
도시: 제주도
날씨: 데이터 없음, 약 20°C 예상
추천: ✅ 추천
팁:
  - 현재 상세 기상 데이터는 없지만, 약 20°C로 예상되어 대체로 여행하기 무난한 기온입니다.
  - 제주도는 바람이 강하고 날씨 변화가 잦아 얇은 겉옷을 꼭 챙기세요.
  - 비 가능성에 대비해 우산이나 방수 재킷을 준비하면 좋습니다.
  - 야외 일정이 많다면 아침에 최신 일기예보를 한 번 더 확인하세요.
  - 해안·오름 방문 시 체감온도가 더 낮을 수 있으니 편한 운동화와 바람막이를 추천합니다.


## 정리: PydanticAI 핵심 패턴 요약

| 개념 | 역할 | 코드 |
|------|------|------|
| **Agent** | LLM을 감싸는 핵심 객체 | `Agent("openai:gpt-5.4")` |
| **instructions** | 역할·규칙 부여 (시스템 프롬프트) | `Agent(..., instructions="...")` |
| **output_type** | 응답을 Pydantic 모델로 강제 | `Agent(..., output_type=MyModel)` |
| **@agent.tool_plain** | LLM이 호출할 수 있는 함수 등록 | `@agent.tool_plain` 데코레이터 |
| **await agent.run()** | Agent 실행 (Jupyter용) | `result = await agent.run("...")` |
| **result.output** | 응답 데이터 접근 | `result.output.title` |

### 다음 단계

이 개념들이 실전에서 어떻게 조합되는지 확인해보세요:

| 노트북 | Agent 유형 | 핵심 패턴 |
|--------|-----------|----------|
| `analysis_agent.ipynb` | 분석형 | output_type만 사용 (Tool 없음) |
| `automation_agent.ipynb` | 자동화형 | Tool + output_type (고정 순서) |
| `planner_agent.ipynb` | Planner형 | Tool만 사용 (LLM이 순서 결정) |